# Creating sample data for logging

In [1]:
import csv
import io

RAW = """id,name,age,salary
1,Alice,30,85000
2,Bob,,62000
3,Carol,41,not_a_number
4,Dave,29,58000
4,Dave,29,58000
5,Eve,230,91000
6,Frank,35,70000
"""

rows = list(csv.DictReader(io.StringIO(RAW)))
print(rows[:2])

[{'id': '1', 'name': 'Alice', 'age': '30', 'salary': '85000'}, {'id': '2', 'name': 'Bob', 'age': '', 'salary': '62000'}]


In [2]:
import logging

def build_logger(name, logfile="pipeline.log"):
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)             

    logger.propagate = False  # Loggers form a family tree.
    # When your "pipeline" logger handles a message, it first gives it to its own handlers.
    # Then, by default, it passes a copy up to its parent, the root logger,
    # which may have its own handlers.
    # Without .propogate = False -
    # log.info("hi")
    #    ├─> pipeline's console handler  -> prints "hi"
    #    └─> (propagates up) root's handler -> prints "hi" again

    logger.handlers.clear() # When you re-run the cell (which you will do a lot in a notebook),
    # addHandler adds another console handler and another file handler
    # handlers.clear() removes the old handlers at the start
    # of the function, so every run starts with exactly one console handler
    # and one file handler.                  

    fmt = logging.Formatter(                   
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    ) # How log lines appear

    console = logging.StreamHandler()          
    console.setLevel(logging.INFO)
    console.setFormatter(fmt)

    file = logging.FileHandler(logfile, mode="w")   
    file.setLevel(logging.DEBUG)
    file.setFormatter(fmt)

    logger.addHandler(console)                 
    logger.addHandler(file)

    return logger

log = build_logger("pipeline")
log.debug("only in the file")
log.info("in the console and the file")

2026-10-04 03:39:18,954 - pipeline - INFO - in the console and the file


In [3]:
def load(text):
    log.info("Loading data")
    rows = list(csv.DictReader(io.StringIO(text)))
    log.debug("Loaded %d raw rows; columns=%s", len(rows), list(rows[0]))
    return rows


def clean(rows):
    seen, good = set(), []
    for row in rows:
        rid = row["id"]

        if rid in seen:
            log.warning("Duplicate id=%s dropped", rid)
            continue
        seen.add(rid)

        try:
            if not row["age"]:
                raise ValueError("missing age")
            row["age"] = int(row["age"])
            row["salary"] = float(row["salary"])
        except ValueError:
            log.exception("id=%s failed parsing, dropped", rid)
            continue

        if not 0 < row["age"] < 120:
            log.error("id=%s has implausible age=%s, dropped", rid, row["age"])
            continue

        log.debug("id=%s ok", rid)
        good.append(row)
    return good


def summarize(rows):
    if not rows:
        log.critical("No valid rows left; cannot summarize")
        raise RuntimeError("empty dataset")
    avg = sum(r["salary"] for r in rows) / len(rows)
    log.info("Rows kept: %d | average salary: %.2f", len(rows), avg)
    return avg

In [4]:
rows = load(RAW)
clean_rows = clean(rows)
summarize(clean_rows)
log.info("Dropped %d of %d rows", len(rows) - len(clean_rows), len(rows))

print(open("pipeline.log").read())

2026-10-04 03:39:18,962 - pipeline - INFO - Loading data
2026-10-04 03:39:18,962 - pipeline - ERROR - id=2 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 20, in clean
    raise ValueError("missing age")
ValueError: missing age
2026-10-04 03:39:18,963 - pipeline - ERROR - id=3 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 22, in clean
    row["salary"] = float(row["salary"])
                    ^^^^^^^^^^^^^^^^^^^^
ValueError: could not convert string to float: 'not_a_number'
2026-10-04 03:39:18,963 - pipeline - WARNING - Duplicate id=4 dropped
2026-10-04 03:39:18,967 - pipeline - ERROR - id=5 has implausible age=230, dropped
2026-10-04 03:39:18,968 - pipeline - INFO - Rows kept: 3 | average salary: 71000.00
2026-10-04 03:39:18,969 - pipeline - INFO - Dropped 4 of 7 rows


2026-10-04 03:39:18,954 - pipeline - DEBUG - only in the file
2026-10-04 03:39:18,954 - pipeline - INFO - in the console and the file
2026-10-04 03:39:18,962 - pipeline - INFO - Loading data
2026-10-04 03:39:18,962 - pipeline - DEBUG - Loaded 7 raw rows; columns=['id', 'name', 'age', 'salary']
2026-10-04 03:39:18,962 - pipeline - DEBUG - id=1 ok
2026-10-04 03:39:18,962 - pipeline - ERROR - id=2 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 20, in clean
    raise ValueError("missing age")
ValueError: missing age
2026-10-04 03:39:18,963 - pipeline - ERROR - id=3 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 22, in clean
    row["salary"] = float(row["salary"])
                    ^^^^^^^^^^^^^^^^^^^^
ValueError: could not convert string to float: 'not_a_number'
2026-10-04 03:39

## No DEBUG logs are printed on the console because the console handler's level is set to INFO, and DEBUG is below INFO. A handler only writes records at or above its level.

In [5]:
import logging
print(logging.DEBUG, logging.INFO, logging.WARNING, logging.ERROR, logging.CRITICAL)
# 10 20 30 40 50

10 20 30 40 50


## Changing console level

In [6]:
for h in log.handlers:
    print(type(h).__name__, h.level)   # find the console handler
log.handlers[0].setLevel(logging.WARNING)   # handlers[0] is the console, because it was added first

StreamHandler 20
FileHandler 10


In [7]:
rows = load(RAW)
clean_rows = clean(rows)
summarize(clean_rows)
log.info("Dropped %d of %d rows", len(rows) - len(clean_rows), len(rows))

print(open("pipeline.log").read())

2026-10-04 03:39:18,995 - pipeline - ERROR - id=2 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 20, in clean
    raise ValueError("missing age")
ValueError: missing age
2026-10-04 03:39:18,995 - pipeline - ERROR - id=3 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 22, in clean
    row["salary"] = float(row["salary"])
                    ^^^^^^^^^^^^^^^^^^^^
ValueError: could not convert string to float: 'not_a_number'
2026-10-04 03:39:18,995 - pipeline - WARNING - Duplicate id=4 dropped
2026-10-04 03:39:18,996 - pipeline - ERROR - id=5 has implausible age=230, dropped


2026-10-04 03:39:18,954 - pipeline - DEBUG - only in the file
2026-10-04 03:39:18,954 - pipeline - INFO - in the console and the file
2026-10-04 03:39:18,962 - pipeline - INFO - Loading data
2026-10-04 03:39:18,962 - pipeline - DEBUG - Loaded 7 raw rows; columns=['id', 'name', 'age', 'salary']
2026-10-04 03:39:18,962 - pipeline - DEBUG - id=1 ok
2026-10-04 03:39:18,962 - pipeline - ERROR - id=2 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 20, in clean
    raise ValueError("missing age")
ValueError: missing age
2026-10-04 03:39:18,963 - pipeline - ERROR - id=3 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 22, in clean
    row["salary"] = float(row["salary"])
                    ^^^^^^^^^^^^^^^^^^^^
ValueError: could not convert string to float: 'not_a_number'
2026-10-04 03:39

## Empty dataset

In [8]:
def load(text):
    log.info("Loading data")
    rows = list(csv.DictReader(io.StringIO(text)))
    if not rows:
        log.warning("Input has a header but no data rows")
        return rows
    log.debug("Loaded %d raw rows; columns=%s", len(rows), list(rows[0]))
    return rows

In [9]:
EMPTY = "id,name,age,salary\n"

rows = load(EMPTY)
clean_rows = clean(rows)

try:
    summarize(clean_rows)
except RuntimeError:
    log.exception("Pipeline aborted")

print(open("pipeline.log").read())

2026-10-04 03:39:19,003 - pipeline - WARNING - Input has a header but no data rows
2026-10-04 03:39:19,004 - pipeline - CRITICAL - No valid rows left; cannot summarize
2026-10-04 03:39:19,004 - pipeline - ERROR - Pipeline aborted
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/688507755.py", line 7, in <module>
    summarize(clean_rows)
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 39, in summarize
    raise RuntimeError("empty dataset")
RuntimeError: empty dataset


2026-10-04 03:39:18,954 - pipeline - DEBUG - only in the file
2026-10-04 03:39:18,954 - pipeline - INFO - in the console and the file
2026-10-04 03:39:18,962 - pipeline - INFO - Loading data
2026-10-04 03:39:18,962 - pipeline - DEBUG - Loaded 7 raw rows; columns=['id', 'name', 'age', 'salary']
2026-10-04 03:39:18,962 - pipeline - DEBUG - id=1 ok
2026-10-04 03:39:18,962 - pipeline - ERROR - id=2 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 20, in clean
    raise ValueError("missing age")
ValueError: missing age
2026-10-04 03:39:18,963 - pipeline - ERROR - id=3 failed parsing, dropped
Traceback (most recent call last):
  File "/var/folders/kx/pd4sg77j7v5fxr5t26brhwqw0000gn/T/ipykernel_83666/4260667832.py", line 22, in clean
    row["salary"] = float(row["salary"])
                    ^^^^^^^^^^^^^^^^^^^^
ValueError: could not convert string to float: 'not_a_number'
2026-10-04 03:39

## Rotating file handlers

### A plain FileHandler appends to one file forever. In a long-running program, such as a web service or a nightly job that runs for months, that file keeps growing. Eventually it fills the disk or becomes too big to open and search. A RotatingFileHandler limits that growth. When the current file reaches a size limit, it:
- Renames the full file to .1 (and shifts older ones to .2, and so on).
- Starts a fresh file.
- Deletes the oldest file once there are more backups than backupCount.

In [10]:
from logging.handlers import RotatingFileHandler

fmt = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

rot = RotatingFileHandler("rotating.log", maxBytes=500, backupCount=2)
rot.setLevel(logging.DEBUG)
rot.setFormatter(fmt)
log.addHandler(rot)

In [11]:
for i in range(30):
    log.debug("filler message number %d", i)

In [12]:
import os, glob

for f in sorted(glob.glob("rotating.log*")):
    print(f, os.path.getsize(f), "bytes")

print("\n--- rotating.log (newest) ---")
print(open("rotating.log").read())

rotating.log 140 bytes
rotating.log.1 490 bytes
rotating.log.2 490 bytes

--- rotating.log (newest) ---
2026-10-04 03:39:19,021 - pipeline - DEBUG - filler message number 28
2026-10-04 03:39:19,021 - pipeline - DEBUG - filler message number 29

